In [ ]:
import numpy as np
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator, Statevector, state_fidelity, random_statevector
import scipy as sp
from qiskit.circuit import QuantumCircuit, QuantumRegister, Parameter
from qiskit.circuit.library import StatePreparation

from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler, QiskitRuntimeService, EstimatorV2 as Estimator
import itertools as it
from typing import Union, List
import matplotlib.pylab as plt
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter, SuzukiTrotter
from collections import Counter

import torch
from NNVQE_HEA import *
import random 

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.qubit import solve_qubit
import warnings

from qiskit_ibm_runtime.fake_provider import FakeSherbrooke 

 
warnings.filterwarnings("ignore")
if __name__ == "__main__":
    import multiprocessing as mp
    mp.set_start_method("spawn", force=True)


# noise sim
backend = FakeSherbrooke()

seed = 1
size = 200

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
def Build_Hamiltonian(n_qubits, edge_data, node_feats):
    """
    circuit: Qiskit QuantumCircuit
    edges:   [(i, j), (k, l), ...] edge list
    edge_data: list of [J_xx, J_yy, J_zz] values for each edge
               must be ordered the same way as edges
    """
    n = n_qubits  # number of qubits used by the circuit
    coeffs = []
    paulis = []

    # Iterate over edges and edge_data together
    for (q1, q2), (J_xx, J_yy, J_zz) in ((k, v.tolist()) for k, v in edge_data.items()):
        # X_q1 X_q2
        pauli_string = ['I'] * n
        pauli_string[q1] = 'X'
        pauli_string[q2] = 'X'
        paulis.append(Pauli(''.join(pauli_string)))
        coeffs.append(J_xx)

        # Y_q1 Y_q2
        pauli_string = ['I'] * n
        pauli_string[q1] = 'Y'
        pauli_string[q2] = 'Y'
        paulis.append(Pauli(''.join(pauli_string)))
        coeffs.append(J_yy)

        # Z_q1 Z_q2
        pauli_string = ['I'] * n
        pauli_string[q1] = 'Z'
        pauli_string[q2] = 'Z'
        paulis.append(Pauli(''.join(pauli_string)))
        coeffs.append(J_zz)

    K_x = node_feats[0][-1].item()
    for i in range(n):
        s = ['I'] * n
        s[i] = 'X'
        paulis.append(Pauli(''.join(s)))
        coeffs.append(K_x)

    # Build the Hamiltonian
    H = SparsePauliOp(paulis, coeffs=coeffs)
    return H

def postselect_counts(counts, num_ones):
    filtered_counts = {}
    for bitstring, freq in counts.items():
        if bitstring.count("1") == num_ones:
            filtered_counts[bitstring] = freq
 
    return filtered_counts

In [ ]:
test_edge_full_data = torch.load('edge_full_data_test.pt')
test_node_full_data = torch.load('node_full_data_test.pt')

In [ ]:
data = torch.randint(low=0, high=len(test_edge_full_data), size=(size,)).tolist()
# data


In [ ]:
shots = 25

num_trotter_steps = 10
krylov_dim = 10

n_qubit=8


res_RND = {}
fidelity_random_initial=[]
res_RND_list = []


sv = random_statevector(2**n_qubit, seed=seed)   # random |psi> from the Haar distribution
initial_state = QuantumCircuit(n_qubit)
initial_state.append(StatePreparation(sv), range(n_qubit))
j=1

for d in data:
    edge_data = test_edge_full_data[d]
    node_feats = test_node_full_data[d]
    edges = list(edge_data.keys())

    H_op = Build_Hamiltonian(n_qubit, edge_data, node_feats)
    H_mat = H_op.to_matrix()
    eigvals, eigvecs = sp.sparse.linalg.eigsh(H_mat, which='SA', k=2)
    eigvals = eigvals.real
    ground_state = eigvecs[:, np.argmin(eigvals)]
    exact_gs_en = np.min(eigvals)

    norm_H_bound = np.sum(np.abs(H_op.coeffs))
    dt = np.pi/norm_H_bound
    fid = state_fidelity(Statevector(ground_state), sv)
    fidelity_random_initial.append(fid)

    evol_gate = PauliEvolutionGate(H_op, time=(dt / num_trotter_steps), synthesis=LieTrotter(reps=num_trotter_steps))

    qr = QuantumRegister(n_qubit)
    qc_evol = QuantumCircuit(qr)
    qc_evol.append(evol_gate, qargs=qr)
    
    circuits = []
    for rep in range(krylov_dim):
        circ = initial_state.copy()
    
        for _ in range(rep):
            circ.compose(other=qc_evol, inplace=True)
    
        circ.measure_all()
        circuits.append(circ)
    # circuits[1].decompose().draw("mpl", fold=-1)

    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    isa_circuits = pm.run(circuits=circuits)
    sampler = Sampler(mode=backend)
    job = sampler.run(isa_circuits, shots=shots)
    # job.result()[0].data.meas.get_counts()

    counts_all = [job.result()[k].data.meas.get_counts() for k in range(krylov_dim)]

    counts_cumulative = []
    for i in range(krylov_dim):
        counter = Counter()
        for d in counts_all[: i + 1]:
            counter.update(d)
    
        counts = dict(counter)
        counts_cumulative.append(counts)

    scipy_kwargs = {"k": 2, "which": "SA"}
    ground_state_energies = []
    for idx, counts in enumerate(counts_cumulative):
        # counts = postselect_counts(counts, num_ones=n_qubit // 2)
        bitstring_matrix, probs = counts_to_arrays(counts=counts)
    
        eigenvals, eigenstates = solve_qubit(
            bitstring_matrix, H_op, verbose=False, **scipy_kwargs
        )
        gs_en = np.min(eigenvals)
        ground_state_energies.append(gs_en)

    edge_key = tuple(round(x,5) for x in edge_data[(0,1)].tolist())
    node_key = tuple(round(x,5) for x in [node_feats[(0,1)].tolist()])

    gnd_en_circ_list_scale=[]
    for i in range(len(ground_state_energies)):
        gnd_en_circ_list_scale.append(((ground_state_energies[i]-exact_gs_en).item())/np.abs(exact_gs_en).item()) # rounded at the 12th decimal place
    
    print(j)
    print(f"[{edge_key, node_key}]_RND = ", [round(x, 12) for x in gnd_en_circ_list_scale])
    res_RND[edge_key, node_key] = gnd_en_circ_list_scale
    res_RND_list.append(gnd_en_circ_list_scale)

    j+=1

In [ ]:
mean_res_RND = [sum(col) / len(col) for col in zip(*res_RND_list)]
mean_fid = sum(fidelity_random_initial) / len(fidelity_random_initial)


In [ ]:
print("mean_res_RND_1= ", mean_res_RND)
print("Fid_1 = ", mean_fid)

In [ ]:
import pickle
with open("Random_initial_result_1.pkl", "wb") as f:
    pickle.dump(res_RND, f, protocol=pickle.HIGHEST_PROTOCOL)

